In [ ]:
import os
import zipfile
import re
from collections import Counter, defaultdict
from google.colab import files

In [ ]:
# Menampilkan pesan agar pengguna mengunggah file ZIP
print("Upload file ZIP yang berisi dokumen .txt")

# Membuka dialog upload file di Google Colab
uploaded = files.upload()

# Mengambil nama file ZIP yang pertama kali diunggah
nama_zip = list(uploaded.keys())[0]

# Menentukan lokasi folder tujuan ekstraksi
folder_extract = "/content/koleksi_dokumen"

# Membuat folder tujuan jika belum ada
os.makedirs(folder_extract, exist_ok=True)

# Membuka file ZIP dalam mode baca ('r' = read)
with zipfile.ZipFile(nama_zip, 'r') as zip_ref:

    # Mengekstrak seluruh isi file ZIP ke folder tujuan
    zip_ref.extractall(folder_extract)

# Menampilkan pesan bahwa proses ekstraksi berhasil
print(f"\nZIP berhasil diekstrak ke: {folder_extract}")

Upload file ZIP yang berisi dokumen .txt


Saving Koleksi (1).zip to Koleksi (1).zip

ZIP berhasil diekstrak ke: /content/koleksi_dokumen


In [ ]:
# Mengambil semua file dengan ekstensi .txt dari folder hasil ekstraksi
DAFTAR_FILE = sorted([

    # Variabel file akan berisi nama setiap file dalam folder
    file

    # Melakukan perulangan untuk setiap file yang ada di folder_extract
    for file in os.listdir(folder_extract)

    # Hanya memilih file yang berakhiran .txt
    if file.endswith(".txt")
])

# Menyimpan lokasi folder data ke dalam variabel FOLDER_DATA
FOLDER_DATA = folder_extract

# Menampilkan jumlah file .txt yang ditemukan
print(f"Jumlah dokumen ditemukan: {len(DAFTAR_FILE)}")

# Melakukan perulangan untuk setiap file yang ada dalam DAFTAR_FILE
for file in DAFTAR_FILE:

    # Menampilkan nama file satu per satu
    print("-", file)

Jumlah dokumen ditemukan: 12
- doc1.txt
- doc10.txt
- doc11.txt
- doc12.txt
- doc2.txt
- doc3.txt
- doc4.txt
- doc5.txt
- doc6.txt
- doc7.txt
- doc8.txt
- doc9.txt


In [ ]:
# PREPROCESSING DOKUMEN

# Daftar stopword (kata yang dianggap kurang penting dan akan dibuang)
KATA_BUANG = {
    "dan","yang","di","ke","dari","pada","untuk","dengan",
    "atau","itu","ini","karena","sebagai","dalam","juga",
    "akan","seperti","tidak","pun","tersebut","ada",
    "bahwa","hingga","saat","telah","sudah"
}

# Fungsi stemming sederhana untuk menghapus beberapa akhiran umum bahasa Indonesia
def stemming_sederhana(kata):

    # Daftar akhiran yang akan dihilangkan
    akhiran = ["lah", "kah", "nya", "pun", "ku", "mu"]

    # Memeriksa setiap akhiran dalam daftar
    for a in akhiran:

        # Jika kata berakhir dengan akhiran tersebut
        # dan panjang kata masih cukup setelah akhiran dihapus
        if kata.endswith(a) and len(kata) > len(a) + 2:

            # Menghapus akhiran dari kata
            return kata[:-len(a)]

    # Jika tidak ada akhiran yang cocok, kembalikan kata asli
    return kata


# Fungsi utama preprocessing teks
def preprocessing(teks):

    # Mengubah seluruh huruf menjadi huruf kecil (case folding)
    teks = teks.lower()

    # Menghapus tanda baca dan karakter selain huruf, angka, dan spasi
    teks = re.sub(r"[^a-z0-9\s]", " ", teks)

    # Memecah teks menjadi daftar kata (tokenisasi)
    token = teks.split()

    # Menghapus kata-kata yang termasuk dalam stopword
    token = [t for t in token if t not in KATA_BUANG]

    # Melakukan stemming pada setiap token
    token = [stemming_sederhana(t) for t in token]

    # Mengembalikan hasil preprocessing berupa daftar token
    return token

In [ ]:
# SISTEM INVERTED INDEX

class MesinPencarian:

    # Konstruktor class
    def __init__(self):

        # Menyimpan inverted index
        # Format:
        # term -> [{doc_id:1, freq:3}, {doc_id:2, freq:1}]
        self.index = defaultdict(list)

        # Menyimpan mapping ID dokumen ke nama file
        self.id_ke_file = {}

    # Membangun inverted index dari seluruh dokumen
    def bangun_index(self):

        print("="*60)
        print("MEMBANGUN INVERTED INDEX")
        print("="*60)

        # Struktur sementara
        # term -> {doc_id : frekuensi}
        struktur_sementara = defaultdict(dict)

        # Menghitung jumlah dokumen yang berhasil dibaca
        jumlah_dokumen = 0

        # Membaca semua file txt
        for nomor, nama_file in enumerate(DAFTAR_FILE, start=1):

            # Membentuk path lengkap file
            path_file = os.path.join(FOLDER_DATA, nama_file)

            # Jika file tidak ada
            if not os.path.exists(path_file):
                print(f"File {nama_file} tidak ditemukan")
                continue

            # Menyimpan ID dan nama file
            self.id_ke_file[nomor] = nama_file

            # Membaca isi dokumen
            with open(path_file, "r", encoding="utf-8") as f:
                isi = f.read()

            # Preprocessing dokumen
            token = preprocessing(isi)

            # Menghitung frekuensi tiap kata
            frekuensi = Counter(token)

            # Memasukkan ke struktur sementara
            for term, freq in frekuensi.items():
                struktur_sementara[term][nomor] = freq

            jumlah_dokumen += 1

        # Mengubah struktur sementara menjadi posting list
        for term, dokumen in struktur_sementara.items():

            posting = []

            # Membentuk posting list
            for doc_id, freq in dokumen.items():

                posting.append({
                    "doc_id": doc_id,
                    "freq": freq
                })

            # Mengurutkan berdasarkan ID dokumen
            posting.sort(key=lambda x: x["doc_id"])

            # Menyimpan ke inverted index
            self.index[term] = posting

        print(f"Dokumen terbaca : {jumlah_dokumen}")
        print(f"Jumlah term     : {len(self.index)}")

    # PENCARIAN SATU KATA
    def cari_keyword(self, keyword):

        # Preprocessing query
        hasil = preprocessing(keyword)

        if not hasil:
            return "", []

        # Ambil kata pertama
        term = hasil[0]

        # Ambil posting list term
        return term, self.index.get(term, [])

    # PENCARIAN BANYAK KATA
    def cari_banyak(self, query, operator="AND"):

        # Preprocessing query
        terms = preprocessing(query)

        if not terms:
            return [], []

        daftar_posting = []

        # Mengambil posting list setiap term
        for t in terms:
            daftar_posting.append(self.index.get(t, []))

        # Menyimpan skor dokumen
        skor = defaultdict(int)

        # Jika operator AND
        if operator.upper() == "AND":

            kandidat = None

            # Cari irisan dokumen
            for posting in daftar_posting:

                dok = {p["doc_id"] for p in posting}

                if kandidat is None:
                    kandidat = dok
                else:
                    kandidat &= dok

            # Jika tidak ada hasil
            if not kandidat:
                return terms, []

            # Hitung skor total frekuensi
            for posting in daftar_posting:
                for p in posting:
                    if p["doc_id"] in kandidat:
                        skor[p["doc_id"]] += p["freq"]

        # Jika operator OR
        else:

            # Semua dokumen yang mengandung salah satu term
            for posting in daftar_posting:
                for p in posting:
                    skor[p["doc_id"]] += p["freq"]

        # Urutkan berdasarkan skor tertinggi
        hasil = sorted(
            skor.items(),
            key=lambda x: x[1],
            reverse=True
        )

        return terms, hasil

    # MENAMPILKAN ISI INDEX
    def tampilkan_index(self, batas=20):

        print("\nContoh isi inverted index:\n")

        # Menampilkan sejumlah term pertama
        for i, (term, posting) in enumerate(self.index.items()):

            if i >= batas:
                break

            isi = []

            # Membentuk tampilan posting list
            for p in posting:

                isi.append(
                    f"(Doc {p['doc_id']} : {p['freq']})"
                )

            print(f"{term} -> {', '.join(isi)}")

In [ ]:
# MENU UTAMA PROGRAM

def jalankan():

    # Membuat objek mesin pencarian
    mesin = MesinPencarian()

    # Membangun inverted index dari seluruh dokumen
    mesin.bangun_index()

    # Perulangan utama program
    while True:

        # Menampilkan daftar menu
        print("1. Cari satu kata")
        print("2. Cari banyak kata (AND)")
        print("3. Cari banyak kata (OR)")
        print("4. Lihat sebagian index")
        print("0. Keluar")

        # Validasi input menu
        while True:

            # Meminta pengguna memilih menu
            pilihan = input("\nPilih menu: ")

            # Jika input tidak sesuai
            if pilihan not in ["0", "1", "2", "3", "4"]:
                print("Masukkan angka 0, 1, 2, 3, atau 4")
                continue

            break

        # Keluar dari program
        if pilihan == "0":

            print("Program selesai.")
            break

        # MENU 1 : PENCARIAN SATU KATA
        elif pilihan == "1":

            # Meminta keyword dari pengguna
            kata = input("Masukkan keyword : ").strip()

            # Validasi hanya satu kata
            if len(kata.split()) != 1:
                print("Menu ini hanya menerima SATU kata.")
                print("Gunakan menu 2 (AND) atau menu 3 (OR) untuk banyak kata.")
                continue

            # Melakukan pencarian keyword
            term, hasil = mesin.cari_keyword(kata)

            # Jika tidak ditemukan
            if not hasil:
                print("Dokumen tidak ditemukan.")
                continue

            print("\nHasil Pencarian")

            # Menampilkan hasil pencarian
            for no, item in enumerate(hasil, start=1):

                print(
                    f"{no}. "
                    f"{mesin.id_ke_file[item['doc_id']]} "
                    f"(Banyak kata ={item['freq']})"
                )

        # MENU 2 : PENCARIAN AND
        elif pilihan == "2":

            # Meminta query dari pengguna
            q = input("Masukkan query : ")

            # Melakukan pencarian AND
            terms, hasil = mesin.cari_banyak(q, "AND")

            print("\nTerm:", terms)

            # Jika tidak ada hasil
            if not hasil:
                print("Tidak ada hasil.")
                continue

            # Menampilkan dokumen hasil pencarian
            for no, (doc, skor) in enumerate(hasil, start=1):

                print(
                    f"{no}. "
                    f"{mesin.id_ke_file[doc]} "
                    f"(Banyak kata ={skor})"
                )

        # MENU 3 : PENCARIAN OR
        elif pilihan == "3":

            # Meminta query dari pengguna
            q = input("Masukkan query : ")

            # Melakukan pencarian OR
            terms, hasil = mesin.cari_banyak(q, "OR")

            print("\nTerm:", terms)

            # Jika tidak ditemukan
            if not hasil:
                print("Tidak ada hasil.")
                continue

            # Menampilkan hasil pencarian
            for no, (doc, skor) in enumerate(hasil, start=1):

                print(
                    f"{no}. "
                    f"{mesin.id_ke_file[doc]} "
                    f"(Banyak kata ={skor})"
                )

        # MENU 4 : MENAMPILKAN INVERTED INDEX
        elif pilihan == "4":

            # Menampilkan sebagian isi index
            mesin.tampilkan_index()

        # Pengaman jika ada menu yang tidak tersedia
        else:
            print("Menu tidak tersedia.")


# Menjalankan program hanya jika file dieksekusi langsung
if __name__ == "__main__":
    jalankan()

MEMBANGUN INVERTED INDEX
Dokumen terbaca : 12
Jumlah term     : 1430
1. Cari satu kata
2. Cari banyak kata (AND)
3. Cari banyak kata (OR)
4. Lihat sebagian index
0. Keluar
Menu ini hanya menerima SATU kata.
Gunakan menu 2 (AND) atau menu 3 (OR) untuk banyak kata.
1. Cari satu kata
2. Cari banyak kata (AND)
3. Cari banyak kata (OR)
4. Lihat sebagian index
0. Keluar

Hasil Pencarian
1. doc1.txt (Banyak kata =7)
2. doc12.txt (Banyak kata =6)
3. doc3.txt (Banyak kata =14)
4. doc5.txt (Banyak kata =1)
5. doc6.txt (Banyak kata =4)
1. Cari satu kata
2. Cari banyak kata (AND)
3. Cari banyak kata (OR)
4. Lihat sebagian index
0. Keluar

Term: ['semarang', 'api', 'kembang', 'perayaan']
1. doc1.txt (Banyak kata =21)
1. Cari satu kata
2. Cari banyak kata (AND)
3. Cari banyak kata (OR)
4. Lihat sebagian index
0. Keluar

Term: ['semarang', 'api', 'kembang', 'perayaan']
1. doc3.txt (Banyak kata =30)
2. doc1.txt (Banyak kata =21)
3. doc6.txt (Banyak kata =10)
4. doc12.txt (Banyak kata =8)
5. doc5.txt (